# S6E7 — v8 : TE-XGBoost non pondéré + correction beta (recette souvik corrigée)

Enseignements des ablations de souvik (0.95+) :

- **Pas de class weights à l'entraînement** — ça distord les probas sur lesquelles beta s'appuie. La prior-correction se fait UNIQUEMENT via `proba / prior**beta` post-hoc
- **TargetEncoder sklearn** (multiclass, smooth auto) avec encodage OOF interne
- Features brutes (NaN natifs XGB), pas de stats de ligne, pas de lookup, pas de missing flags
- Seed-bag 2 seeds × 5 folds, XGBoost cuda + CatBoost GPU (aussi non pondéré), beta calibré sur OOF

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.metrics import balanced_accuracy_score
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
N_FOLDS = 5
SEEDS = [42, 1]

CANDIDATES = ["/kaggle/input/competitions/playground-series-s6e7",
              "/kaggle/input/playground-series-s6e7", "./data", "."]
DATA_DIR = next(p for p in CANDIDATES if os.path.exists(os.path.join(p, "train.csv")))
print("DATA_DIR:", DATA_DIR)

In [ ]:
TARGET = "health_condition"; ID_COL = "id"
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

cat_cols = [c for c in train.columns if train[c].dtype == "object" and c != TARGET]
num_cols = [c for c in train.columns if c not in [TARGET, ID_COL] + cat_cols]

target_le = LabelEncoder()
y = target_le.fit_transform(train[TARGET])
N_CLASSES = len(target_le.classes_)
prior = np.bincount(y) / len(y)
print("classes:", list(target_le.classes_), "| prior:", prior.round(4))

X = train[num_cols + cat_cols].copy()
X_test = test[num_cols + cat_cols].copy()
print(X.shape, X_test.shape)

In [ ]:
# ---------- XGBoost sur TargetEncoder sklearn, NON PONDÉRÉ ----------
oof_xgb = np.zeros((len(X), N_CLASSES))
tst_xgb = np.zeros((len(X_test), N_CLASSES))
n_runs = N_FOLDS * len(SEEDS)

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        enc = TargetEncoder(target_type="multiclass", smooth="auto",
                            random_state=seed, cv=5)
        Xtr = np.hstack([X.iloc[tr][num_cols].to_numpy(float),
                         enc.fit_transform(X.iloc[tr][cat_cols], y[tr])])
        Xva = np.hstack([X.iloc[va][num_cols].to_numpy(float),
                         enc.transform(X.iloc[va][cat_cols])])
        Xte = np.hstack([X_test[num_cols].to_numpy(float),
                         enc.transform(X_test[cat_cols])])
        m = xgb.XGBClassifier(
            objective="multi:softprob", num_class=N_CLASSES,
            tree_method="hist", device="cuda",
            n_estimators=2000, learning_rate=0.05, max_depth=6, min_child_weight=1.0,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            early_stopping_rounds=200, random_state=seed, n_jobs=-1)
        m.fit(Xtr, y[tr], eval_set=[(Xva, y[va])], verbose=False)
        oof_xgb[va] += m.predict_proba(Xva) / len(SEEDS)
        tst_xgb += m.predict_proba(Xte) / n_runs
        b = balanced_accuracy_score(y[va], m.predict_proba(Xva).argmax(1))
        print(f"[XGB s{seed}] fold {fold+1} | iter={m.best_iteration} | bal_acc={b:.5f}")
print("[XGB] OOF:", round(balanced_accuracy_score(y, oof_xgb.argmax(1)), 5))

In [ ]:
# ---------- CatBoost natif, NON PONDÉRÉ ----------
for c in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([X[c], X_test[c]]).astype(str))
    X[c + "_code"] = le.transform(X[c].astype(str))
    X_test[c + "_code"] = le.transform(X_test[c].astype(str))

cb_cols = num_cols + [c + "_code" for c in cat_cols]
cb_cat = [c + "_code" for c in cat_cols]
df_tr = X[cb_cols].copy(); df_te = X_test[cb_cols].copy()

oof_cat = np.zeros((len(X), N_CLASSES))
tst_cat = np.zeros((len(X_test), N_CLASSES))
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr, va) in enumerate(skf.split(df_tr, y)):
        m = CatBoostClassifier(
            loss_function="MultiClass", classes_count=N_CLASSES,
            iterations=5000, learning_rate=0.04, depth=8, l2_leaf_reg=3.0,
            cat_features=cb_cat, task_type="GPU", devices="0",
            random_seed=seed, verbose=False, od_type="Iter", od_wait=250)
        m.fit(df_tr.iloc[tr], y[tr], eval_set=(df_tr.iloc[va], y[va]), use_best_model=True)
        oof_cat[va] += m.predict_proba(df_tr.iloc[va]) / len(SEEDS)
        tst_cat += m.predict_proba(df_te) / n_runs
        b = balanced_accuracy_score(y[va], m.predict_proba(df_tr.iloc[va]).argmax(1))
        print(f"[CAT s{seed}] fold {fold+1} | iter={m.best_iteration_} | bal_acc={b:.5f}")
print("[CAT] OOF:", round(balanced_accuracy_score(y, oof_cat.argmax(1)), 5))

In [ ]:
def bacc(p):
    return balanced_accuracy_score(y, p.argmax(1))

best_w, best_s = 0.5, 0.0
for w in np.arange(0.0, 1.01, 0.05):
    s = bacc(w * oof_xgb + (1 - w) * oof_cat)
    if s > best_s:
        best_s, best_w = s, w
print(f"blend xgb={best_w:.2f} cat={1-best_w:.2f} | OOF = {best_s:.5f}")

oof = best_w * oof_xgb + (1 - best_w) * oof_cat
tst = best_w * tst_xgb + (1 - best_w) * tst_cat

BETAS = np.arange(0.0, 1.51, 0.05)
scores = {bb: balanced_accuracy_score(y, (oof / prior ** bb).argmax(1)) for bb in BETAS}
best_beta = max(scores, key=scores.get)
print(f"beta optimal = {best_beta:.2f} | OOF {scores[0.0]:.5f} -> {scores[best_beta]:.5f}")

np.save("oof.npy", oof); np.save("test_proba.npy", tst)
with open("beta.txt", "w") as f:
    f.write(str(best_beta))

pred = target_le.inverse_transform((tst / prior ** best_beta).argmax(1))
submission = sample.copy()
submission[TARGET] = pred
submission.to_csv("submission.csv", index=False)
print(submission[TARGET].value_counts(normalize=True))
print("OK submission.csv")